# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection


In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [14:52<00:00, 178.50s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

"Title: Bella Pro MasterBlend 3-in-1 Prep System for $50 + free shipping\nDetails: Today only, this is $120 off and at best price we've seen. We previously listed it for $25 more. Buy Now at Best Buy\nFeatures: 1,200W motor 7 blending and two food processing functions 6-point stainless steel blade Model: 90217\nURL: https://www.dealnews.com/products/Bella/Bella-Pro-Master-Blend-3-in-1-Prep-System/487822.html?iref=rss-c196"

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Unlocked Google Pixel Fold 256GB Android Smartphone for $600 + free shipping
Details: It's an all-time price low for this unlocked version of the 256GB Google Pixel Fold, which was released in 2023. You'd pay $150 more for this phone at Amazon today.This deal was good enough to make our roundup of the top five best deals of the day. Follow that link to see our lates

In [9]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [10]:
result = get_recommendations()

In [11]:
len(result.deals)

5

In [12]:
result.deals[1]

Deal(product_description='The Refurbished Apple Watch Series 6 features a 40mm display and comes with GPS + Cellular capabilities. It includes an Always-On Retina display and tracks health metrics such as blood oxygen levels, making it ideal for fitness enthusiasts. With a battery life of up to 18 hours, this watch allows for all-day use without frequent charging. The model is known for its elegance paired with functionality.', price=107.0, url='https://www.dealnews.com/products/Apple/Apple-Watch-Series-6-40-mm-GPS-Cellular-Smartwatch/172105.html?iref=rss-c142')

In [13]:
from agents.scanner_agent import ScannerAgent

In [ ]:
agent = ScannerAgent()
result = agent.scan()

In [ ]:
result